In [32]:
# Install required libraries
!pip install google-generativeai pandas openpyxl sqlparse

In [33]:
!pip install groq

In [34]:
import os
import pandas as pd
import sqlparse
import re
import time
from groq import Groq

# Paste your NEW Groq API key below between the quotes
GROQ_API_KEY = "API KEY HERE "

client = Groq(api_key=GROQ_API_KEY)

def ask_ai(prompt):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1000
    )
    return response.choices[0].message.content

# Test connection
test = ask_ai("Say: AI Connected Successfully!")
print(f"✅ {test}")

✅ **AI Connected Successfully**


In [35]:
def extract_tables_from_sql(sql_text):
    tables = {}

    # Find all CREATE TABLE blocks
    pattern = re.findall(
        r'CREATE\s+TABLE\s+[`"]?(\w+)[`"]?\s*\((.*?)\);',
        sql_text,
        re.IGNORECASE | re.DOTALL
    )

    for table_name, columns_block in pattern:
        columns = []
        for line in columns_block.strip().split('\n'):
            line = line.strip().rstrip(',')
            # Skip constraints and keys
            if any(skip in line.upper() for skip in ['PRIMARY', 'FOREIGN', 'KEY', 'INDEX', 'UNIQUE', 'CONSTRAINT']):
                continue
            if line:
                # Extract column name and type
                parts = line.split()
                if len(parts) >= 2:
                    col_name = parts[0].strip('`"')
                    col_type = parts[1].strip('`"')
                    columns.append({
                        'column_name': col_name,
                        'data_type': col_type
                    })

        if columns:
            tables[table_name] = columns

    return tables

print("✅ SQL Parser ready!")

✅ SQL Parser ready!


In [36]:
def analyze_table_with_ai(table_name, columns):
    column_list = "\n".join([
        f"  - {col['column_name']} ({col['data_type']})"
        for col in columns
    ])

    prompt = f"""You are a senior Fortune 500 data architect. Analyze this database table strictly.

Table Name: {table_name}

Columns:
{column_list}

Respond in EXACTLY this format with no markdown, no asterisks, no bold:

TABLE PURPOSE: Write exactly 2 sentences about what this table stores.

COMPLEXITY SCORE: Give a single number from 1 to 10 only.

BUSINESS DOMAIN: Choose exactly one from: Customer, Finance, Inventory, Orders, HR, Marketing, Support, Transactions

KEY COLUMNS: List the 2 most important columns and one sentence why.

COLUMN DESCRIPTIONS:
{chr(10).join([f"{col['column_name']}: One clear sentence describing this column." for col in columns])}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1000,
        temperature=0.3,
    )
    return response.choices[0].message.content

In [37]:
from openpyxl import Workbook
from openpyxl.styles import (PatternFill, Font, Alignment, Border, Side)
from openpyxl.utils import get_column_letter

def generate_excel_report(results, filename="AI_Metadata_Report.xlsx"):
    wb = Workbook()

    ws1 = wb.active
    ws1.title = "Executive Summary"

    dark_blue  = PatternFill("solid", fgColor="1B2A4A")
    mid_blue   = PatternFill("solid", fgColor="2E5FA3")
    light_blue = PatternFill("solid", fgColor="D6E4F0")
    green_fill = PatternFill("solid", fgColor="1E8449")
    yellow     = PatternFill("solid", fgColor="F4D03F")
    red_fill   = PatternFill("solid", fgColor="C0392B")

    white_bold  = Font(color="FFFFFF", bold=True, size=12)
    dark_font   = Font(color="1B2A4A", bold=True, size=11)
    normal_font = Font(color="1B2A4A", size=10)

    thin = Side(style="thin", color="AAAAAA")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    ws1.merge_cells("A1:F1")
    ws1["A1"] = "🧠 AI Metadata Intelligence Engine — Executive Summary"
    ws1["A1"].font = Font(color="FFFFFF", bold=True, size=16)
    ws1["A1"].fill = dark_blue
    ws1["A1"].alignment = Alignment(horizontal="center", vertical="center")
    ws1.row_dimensions[1].height = 40

    ws1.merge_cells("A2:F2")
    ws1["A2"] = "Generated by AI Metadata Intelligence Engine | Powered by Groq AI"
    ws1["A2"].font = Font(color="FFFFFF", size=10, italic=True)
    ws1["A2"].fill = mid_blue
    ws1["A2"].alignment = Alignment(horizontal="center")

    ws1.merge_cells("A4:F4")
    ws1["A4"] = "DATABASE OVERVIEW"
    ws1["A4"].font = white_bold
    ws1["A4"].fill = mid_blue
    ws1["A4"].alignment = Alignment(horizontal="center")

    total_tables  = len(results)
    total_columns = sum(r['total_columns'] for r in results)
    avg_cols      = round(total_columns / total_tables, 1) if total_tables else 0

    stats = [
        ("Total Tables Analyzed",  total_tables,  "🗄️"),
        ("Total Columns Mapped",   total_columns, "📊"),
        ("Avg Columns per Table",  avg_cols,      "📈"),
        ("AI Descriptions Generated", total_columns + total_tables, "🤖"),
    ]

    for col_idx, (label, value, icon) in enumerate(stats, 1):
        cell_label = ws1.cell(row=5, column=col_idx, value=f"{icon} {label}")
        cell_label.fill  = light_blue
        cell_label.font  = dark_font
        cell_label.alignment = Alignment(horizontal="center", wrap_text=True)
        cell_label.border = border

        cell_value = ws1.cell(row=6, column=col_idx, value=value)
        cell_value.fill  = PatternFill("solid", fgColor="FFFFFF")
        cell_value.font  = Font(color="2E5FA3", bold=True, size=20)
        cell_value.alignment = Alignment(horizontal="center")
        cell_value.border = border

    ws1.row_dimensions[6].height = 40

    headers = ["Table Name", "Total Columns", "Business Domain", "Complexity Score", "Status"]
    for col_idx, header in enumerate(headers, 1):
        cell = ws1.cell(row=8, column=col_idx, value=header)
        cell.fill      = dark_blue
        cell.font      = white_bold
        cell.alignment = Alignment(horizontal="center")
        cell.border    = border

    for row_idx, result in enumerate(results, 9):
        analysis = result['ai_analysis']

        # ✅ FIXED: Now correctly reads BUSINESS DOMAIN
        domain = "General"
        for line in analysis.split('\n'):
            if 'BUSINESS DOMAIN:' in line:
                domain = line.replace('BUSINESS DOMAIN:', '').strip()
                break

        score = 5
        score_match = re.search(r'COMPLEXITY SCORE[:\s]+(\d+)', analysis)
        if score_match:
            score = int(score_match.group(1))

        if score <= 3:
            score_fill = green_fill
        elif score <= 6:
            score_fill = yellow
        else:
            score_fill = red_fill

        row_fill = PatternFill("solid", fgColor="F8FBFF" if row_idx % 2 == 0 else "FFFFFF")

        values = [result['table_name'], result['total_columns'], domain, score, "✅ Documented"]

        for col_idx, value in enumerate(values, 1):
            cell = ws1.cell(row=row_idx, column=col_idx, value=value)
            cell.fill      = score_fill if col_idx == 4 else row_fill
            cell.font      = Font(color="FFFFFF", bold=True) if col_idx == 4 else normal_font
            cell.alignment = Alignment(horizontal="center")
            cell.border    = border

    for col, width in zip("ABCDEF", [30, 15, 25, 20, 20, 20]):
        ws1.column_dimensions[get_column_letter(ord(col)-64)].width = width

    ws2 = wb.create_sheet("Detailed Analysis")

    ws2.merge_cells("A1:D1")
    ws2["A1"] = "📋 Detailed AI Analysis — All Tables"
    ws2["A1"].font  = Font(color="FFFFFF", bold=True, size=14)
    ws2["A1"].fill  = dark_blue
    ws2["A1"].alignment = Alignment(horizontal="center", vertical="center")
    ws2.row_dimensions[1].height = 35

    detail_headers = ["Table Name", "Column Name", "Data Type", "AI Description"]
    for col_idx, header in enumerate(detail_headers, 1):
        cell = ws2.cell(row=2, column=col_idx, value=header)
        cell.fill      = mid_blue
        cell.font      = white_bold
        cell.alignment = Alignment(horizontal="center")
        cell.border    = border

    current_row = 3
    for result in results:
        analysis_text = result['ai_analysis']

        col_descriptions = {}
        lines = analysis_text.split('\n')
        for line in lines:
            for col in result['columns']:
                if col['column_name'].lower() in line.lower() and ':' in line:
                    desc = line.split(':', 1)[-1].strip()
                    if len(desc) > 10:
                        col_descriptions[col['column_name']] = desc

        table_fill = PatternFill("solid", fgColor="EBF5FB")

        for col in result['columns']:
            description = col_descriptions.get(
                col['column_name'],
                "Auto-mapped column — see AI analysis sheet"
            )

            row_data = [result['table_name'], col['column_name'],
                        col['data_type'], description]

            for col_idx, value in enumerate(row_data, 1):
                cell = ws2.cell(row=current_row, column=col_idx, value=value)
                cell.fill      = table_fill if current_row % 2 == 0 else PatternFill("solid", fgColor="FFFFFF")
                cell.font      = normal_font
                cell.alignment = Alignment(horizontal="left", wrap_text=True)
                cell.border    = border

            current_row += 1

    for col, width in zip("ABCD", [25, 25, 15, 60]):
        ws2.column_dimensions[get_column_letter(ord(col)-64)].width = width

    ws3 = wb.create_sheet("Raw AI Analysis")

    ws3.merge_cells("A1:B1")
    ws3["A1"] = "🤖 Full AI Analysis Per Table"
    ws3["A1"].font  = Font(color="FFFFFF", bold=True, size=14)
    ws3["A1"].fill  = dark_blue
    ws3["A1"].alignment = Alignment(horizontal="center", vertical="center")
    ws3.row_dimensions[1].height = 35

    for col_idx, header in enumerate(["Table Name", "Full AI Analysis"], 1):
        cell = ws3.cell(row=2, column=col_idx, value=header)
        cell.fill      = mid_blue
        cell.font      = white_bold
        cell.alignment = Alignment(horizontal="center")
        cell.border    = border

    for row_idx, result in enumerate(results, 3):
        ws3.cell(row=row_idx, column=1, value=result['table_name']).font = dark_font
        analysis_cell = ws3.cell(row=row_idx, column=2, value=result['ai_analysis'])
        analysis_cell.alignment = Alignment(wrap_text=True)
        analysis_cell.font      = normal_font
        ws3.row_dimensions[row_idx].height = 120

    ws3.column_dimensions['A'].width = 25
    ws3.column_dimensions['B'].width = 100

    wb.save(filename)
    print(f"✅ Enterprise Excel Report saved: {filename}")
    return filename

print("✅ Excel Report Generator ready!")

✅ Excel Report Generator ready!


In [38]:
# Creating a sample SQL file to test with
sample_sql = """
CREATE TABLE customers (
    customer_id INT,
    first_name VARCHAR(50),
    last_name VARCHAR(50),
    email VARCHAR(100),
    phone VARCHAR(20),
    date_of_birth DATE,
    registration_date TIMESTAMP,
    loyalty_points INT,
    account_status VARCHAR(20)
);

CREATE TABLE orders (
    order_id INT,
    customer_id INT,
    order_date TIMESTAMP,
    delivery_date DATE,
    total_amount DECIMAL,
    discount_amount DECIMAL,
    payment_method VARCHAR(30),
    order_status VARCHAR(20),
    shipping_address TEXT
);

CREATE TABLE products (
    product_id INT,
    product_name VARCHAR(100),
    category VARCHAR(50),
    brand VARCHAR(50),
    unit_price DECIMAL,
    cost_price DECIMAL,
    stock_quantity INT,
    reorder_level INT,
    supplier_id INT,
    created_date TIMESTAMP
);

CREATE TABLE employees (
    employee_id INT,
    first_name VARCHAR(50),
    last_name VARCHAR(50),
    department VARCHAR(50),
    job_title VARCHAR(100),
    salary DECIMAL,
    hire_date DATE,
    manager_id INT,
    performance_score FLOAT,
    is_active BOOLEAN
);

CREATE TABLE transactions (
    transaction_id INT,
    order_id INT,
    payment_date TIMESTAMP,
    amount DECIMAL,
    currency VARCHAR(10),
    payment_gateway VARCHAR(50),
    transaction_status VARCHAR(20),
    failure_reason VARCHAR(200)
);

CREATE TABLE inventory_log (
    log_id INT,
    product_id INT,
    warehouse_id INT,
    movement_type VARCHAR(20),
    quantity_change INT,
    movement_date TIMESTAMP,
    performed_by INT,
    notes TEXT
);

CREATE TABLE support_tickets (
    ticket_id INT,
    customer_id INT,
    assigned_agent_id INT,
    category VARCHAR(50),
    priority VARCHAR(20),
    subject VARCHAR(200),
    description TEXT,
    created_at TIMESTAMP,
    resolved_at TIMESTAMP,
    satisfaction_score INT
);

CREATE TABLE marketing_campaigns (
    campaign_id INT,
    campaign_name VARCHAR(100),
    channel VARCHAR(50),
    start_date DATE,
    end_date DATE,
    budget DECIMAL,
    spent_amount DECIMAL,
    impressions INT,
    clicks INT,
    conversions INT
);
"""

# Save it as a file
with open("sample_database.sql", "w") as f:
    f.write(sample_sql)

print("✅ Sample SQL file created with 8 tables!")
print("📊 Tables included:")
print("   - customers, orders, products, employees")
print("   - transactions, inventory_log, support_tickets, marketing_campaigns")

✅ Sample SQL file created with 8 tables!
📊 Tables included:
   - customers, orders, products, employees
   - transactions, inventory_log, support_tickets, marketing_campaigns


In [39]:
# ─── FULL ENGINE RUN ───

# Step 1 - Read the SQL file
with open("sample_database.sql", "r") as f:
    sql_text = f.read()

print("✅ SQL file loaded!")

# Step 2 - Extract all tables
tables = extract_tables_from_sql(sql_text)
print(f"✅ Found {len(tables)} tables with their columns!")

# Step 3 - Run AI analysis on every table
results = process_all_tables(tables)

# Step 4 - Generate the Excel report
filename = generate_excel_report(results)

# Step 5 - Download the report
from google.colab import files
files.download(filename)

print("\n🎉 DONE! Your Enterprise Excel Report is downloading!")
print("📊 Report contains:")
print("   Sheet 1 → Executive Summary with complexity scores")
print("   Sheet 2 → Detailed column by column AI descriptions")
print("   Sheet 3 → Full raw AI analysis per table")

✅ SQL file loaded!
✅ Found 8 tables with their columns!
🚀 Starting AI analysis of 8 tables...

⚙️  Analyzing table 1/8: customers
⚙️  Analyzing table 2/8: orders
⚙️  Analyzing table 3/8: products
⚙️  Analyzing table 4/8: employees
⚙️  Analyzing table 5/8: transactions
⚙️  Analyzing table 6/8: inventory_log
⚙️  Analyzing table 7/8: support_tickets
⚙️  Analyzing table 8/8: marketing_campaigns

✅ AI Analysis complete! 8 tables processed.
✅ Enterprise Excel Report saved: AI_Metadata_Report.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 DONE! Your Enterprise Excel Report is downloading!
📊 Report contains:
   Sheet 1 → Executive Summary with complexity scores
   Sheet 2 → Detailed column by column AI descriptions
   Sheet 3 → Full raw AI analysis per table


In [39]:
done got it